In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
import time
import random

### Scraping Kdrama Names and Links

In [32]:
def scrape_pages(start_page, end_page):
    chrome_path = "/Applications/Google Chrome.app/Contents/MacOS/Google Chrome"

    service = Service()  # Selenium will try Selenium Manager again
    options = Options()
    options.binary_location = chrome_path

    driver = webdriver.Chrome(service=service, options=options)

    for page in range(start_page, end_page + 1):
        url = f"https://mydramalist.com/search?adv=titles&ty=68,77,86&co=3&rt=4,10&st=3&so=top&page={page}"
        print(f"Scraping page {page}...")
        
        driver.get(url)

        # wait for JS + Cloudflare
        time.sleep(random.uniform(4, 7))

        titles = driver.find_elements(By.CSS_SELECTOR, "h6.text-primary.title a")
        if not titles:
            print(f"No titles found on page {page}")

        with open("title.txt", "a", encoding="utf-8") as f:
            for title in titles:
                text = title.text.strip()
                link = title.get_attribute('href')
                if text and link:
                    f.write(f"{text} | {link}\n")

        time.sleep(random.uniform(2, 5))

    driver.quit()
    print("Batch complete.\n")

In [33]:
total_pages = 250
batch_size = 10

In [ ]:
for start in range(1, total_pages+1, batch_size):
    end = min(start+batch_size-1,total_pages)
    scrape_pages(start,end)
    time.sleep(random.uniform(10,20))

Scraping page 203...
Scraping page 204...
Scraping page 205...
Scraping page 206...
Scraping page 207...
Scraping page 208...
Scraping page 209...
Scraping page 210...
Scraping page 211...
Scraping page 212...
Batch complete.

Scraping page 213...
Scraping page 214...
Scraping page 215...
Scraping page 216...
Scraping page 217...
Scraping page 218...
Scraping page 219...
Scraping page 220...
Scraping page 221...
Scraping page 222...
Batch complete.

Scraping page 223...
Scraping page 224...
Scraping page 225...
Scraping page 226...
Scraping page 227...
Scraping page 228...
Scraping page 229...
Scraping page 230...
Scraping page 231...
Scraping page 232...
Batch complete.

Scraping page 233...
Scraping page 234...
Scraping page 235...
Scraping page 236...
Scraping page 237...
Scraping page 238...
Scraping page 239...
Scraping page 240...
Scraping page 241...
Scraping page 242...
Batch complete.

Scraping page 243...
Scraping page 244...
Scraping page 245...
Scraping page 246...
Scraping

### Scraping Kdrama Details

In [56]:
import pandas as pd
import requests
import json
import time

In [13]:
movies = pd.read_csv('title.txt',sep='|',names=["Title","Links"])

In [22]:
movies['slug'] = movies['Links'].apply(lambda x: x.split('/')[-1].strip())

In [37]:
movies

,Title,Links,slug
0,Nana Tour with Seventeen,https://mydramalist.com/757373-youth-over-flo...,757373-youth-over-flowers
1,BTS in the Soop Season 2,https://mydramalist.com/710607-bts-in-the-soo...,710607-bts-in-the-soop-season-2
2,Run BTS! 2022 Special Episode: Fly BTS Fly,https://mydramalist.com/740861-run-bts-2022-s...,740861-run-bts-2022-special-episode-fly-bts-fly
3,Suga: Road to D-Day,https://mydramalist.com/750569-suga-road-to-d...,750569-suga-road-to-d-day
4,When Life Gives You Tangerines,https://mydramalist.com/735043-life,735043-life
...,...,...,...
4995,All About Dong Bang Shin Ki Season 2,https://mydramalist.com/51605-all-about-dong-...,51605-all-about-dong-bang-shin-ki-season-2
4996,Chick High Kick,https://mydramalist.com/696167-chick-high-kick,696167-chick-high-kick
4997,Oh My Girl Miracle Expedition,https://mydramalist.com/27014-oh-my-girl-mira...,27014-oh-my-girl-miracle-expedition
4998,Wanna One Amigo TV Season 2,https://mydramalist.com/59553-wanna-one-amigo-tv,59553-wanna-one-amigo-tv


In [24]:
link = 'https://my-drama-list-api-ten.vercel.app/api/id/757373-youth-over-flowers'

In [38]:
movie_database = pd.DataFrame(columns=['slug','url','title','image','synopsis','episodes','aired','content_rating','watchers','genres','tags','ratings'])

In [39]:
movie_database

,slug,url,title,image,synopsis,episodes,aired,content_rating,watchers,genres,tags,ratings


In [44]:
movie_slugs = list(movies.slug)

In [69]:
def write_to_txt(movie_details,output_file,count):
    """helper function: function to write movie details extracted to csv"""
    with open(output_file,'a',encoding='utf-8') as f:
        for md in movie_details:
            f.write(json.dumps(md) + '\n')
        print(f"{count} movie details writen to {output_file}")

In [74]:
def get_movie_details(slugs, output_file = 'Movie_File.jsonl', batch_size = 50):
    """This function gets movie details from dramalist API and uses write_to_txt function to write to a file"""
    session = requests.Session()
    movie_details = []
    for i, slug in enumerate(slugs, 1):
        link = f"https://my-drama-list-api-ten.vercel.app/api/id/{slug}"
        for attempt in range(3):    
            try:
                response = session.get(link, timeout=10)
                response.raise_for_status()
                movie_detail = response.json()
                movie_details.append(movie_detail)
                break
            except requests.exceptions.RequestException:
                print(f"Attempt {attempt + 1} failed for slug {slug}")
                if attempt == 2:
                    print(f"Failed to get movie details for slug {slug}")
                else:
                    time.sleep(0.5)
        if i % batch_size==0:
            write_to_txt(movie_details, output_file,count=i)
            movie_details = []
        time.sleep(0.2)
    if movie_details:
        write_to_txt(movie_details, output_file,count=i)
    print("All done")
        

In [76]:
get_movie_details(slugs=movie_slugs[3949:])

Attempt 1 failed for slug 807-the-doll-master
Attempt 2 failed for slug 807-the-doll-master
Attempt 3 failed for slug 807-the-doll-master
Failed to get movie details for slug 807-the-doll-master
Attempt 1 failed for slug 1552-changing-partners
Attempt 2 failed for slug 1552-changing-partners
Attempt 3 failed for slug 1552-changing-partners
Failed to get movie details for slug 1552-changing-partners
50 movie details writen to Movie_File.jsonl
100 movie details writen to Movie_File.jsonl
Attempt 1 failed for slug 2927-the-intimate-lover
Attempt 2 failed for slug 2927-the-intimate-lover
Attempt 3 failed for slug 2927-the-intimate-lover
Failed to get movie details for slug 2927-the-intimate-lover
150 movie details writen to Movie_File.jsonl
Attempt 1 failed for slug 15165-queer-movie-butterfly-the-adult-world
Attempt 2 failed for slug 15165-queer-movie-butterfly-the-adult-world
Attempt 3 failed for slug 15165-queer-movie-butterfly-the-adult-world
Failed to get movie details for slug 15165-

In [81]:
data = pd.read_json('Movie_File.jsonl',lines = True)

In [82]:
data.to_csv('raw_kdrama.csv')